In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

In [7]:
data = pd.read_csv('train.csv')

In [ ]:
data.head()

,label,integer_feature_1,integer_feature_2,integer_feature_3,integer_feature_4,integer_feature_5,integer_feature_6,integer_feature_7,integer_feature_8,integer_feature_9,...,categorical_feature_17,categorical_feature_18,categorical_feature_19,categorical_feature_20,categorical_feature_21,categorical_feature_22,categorical_feature_23,categorical_feature_24,categorical_feature_25,categorical_feature_26
0,0,49.0,520.0,0.0,353.0,5.0,0.0,0.0,0,2,...,1f7fc70b,628f1b8d,9512c20b,c5362157,a9ad4e8e,2b6a4b2f,bc71cd9d,9f4102b0,ff654802,2f33557c
1,0,8.0,408.0,4.0,32.0,6.0,0.0,0.0,-1,1,...,NaN,a1eb1511,0a1d513f,7fdb0729,747f6ecc,62682372,NaN,401a16fe,30436bfc,b757e957
2,0,28.0,37.0,2.0,5.0,4.0,5.0,1.0,974,14,...,753da5f3,d9f758ff,9512c20b,682d90ba,09afda28,307a0f03,f942364c,16e6b497,57e36578,a7ab145d
3,0,1.0,603.0,1.0,32.0,1.0,0.0,0.0,13,1,...,NaN,b8170bba,108a0699,47849e55,73b3f46d,d994ba60,NaN,328e967b,ff654802,2ba8d787
4,0,8.0,525.0,3.0,8.0,126.0,1.0,0.0,10,11,...,753da5f3,b8170bba,55191c09,0e29b5da,0683bc6f,47b99d80,d2b125b2,00a688ec,30436bfc,b757e957


In [22]:
print(data['categorical_feature_3'].dtype)

str


In [53]:
type(list(data.columns))

list

In [26]:
categorical_feat = [i for i in data.columns if data[i].dtype == 'str']
numerical_feat = [i for i in data.columns if data[i].dtype == float or data[i].dtype == int]

In [30]:
## preprocessing and handle the NAN values

## I put median value from NAN in integer_feature and max occurance for categorical_features

for i in numerical_feat:
    data[i] = data[i].fillna(data[i].median())


for i in categorical_feat:
    data[i] = data[i].fillna(data[i].mode()[0])



In [31]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 40 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   label                   40000 non-null  int64  
 1   integer_feature_1       40000 non-null  float64
 2   integer_feature_2       40000 non-null  float64
 3   integer_feature_3       40000 non-null  float64
 4   integer_feature_4       40000 non-null  float64
 5   integer_feature_5       40000 non-null  float64
 6   integer_feature_6       40000 non-null  float64
 7   integer_feature_7       40000 non-null  float64
 8   integer_feature_8       40000 non-null  int64  
 9   integer_feature_9       40000 non-null  int64  
 10  integer_feature_10      40000 non-null  float64
 11  integer_feature_11      40000 non-null  float64
 12  integer_feature_12      40000 non-null  float64
 13  integer_feature_13      40000 non-null  float64
 14  categorical_feature_1   40000 non-null  str    
 

In [32]:
categorical_data_unique = {}

for i in categorical_feat:
    categorical_data_unique[i] = data[i].unique().shape

In [33]:
categorical_data_unique

{'categorical_feature_1': (13190,),
 'categorical_feature_2': (3757,),
 'categorical_feature_3': (4968,),
 'categorical_feature_4': (1851,),
 'categorical_feature_5': (4253,),
 'categorical_feature_6': (3,),
 'categorical_feature_7': (3111,),
 'categorical_feature_8': (741,),
 'categorical_feature_9': (22,),
 'categorical_feature_10': (11584,),
 'categorical_feature_11': (5568,),
 'categorical_feature_12': (7723,),
 'categorical_feature_13': (9,),
 'categorical_feature_14': (903,),
 'categorical_feature_15': (2025,),
 'categorical_feature_16': (42,),
 'categorical_feature_17': (3,),
 'categorical_feature_18': (341,),
 'categorical_feature_19': (14,),
 'categorical_feature_20': (13508,),
 'categorical_feature_21': (9002,),
 'categorical_feature_22': (12628,),
 'categorical_feature_23': (5094,),
 'categorical_feature_24': (3504,),
 'categorical_feature_25': (36,),
 'categorical_feature_26': (24,)}

In [40]:
## Train test split

X = data.drop(columns=['label'])
y = data['label']

In [42]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state = 42
)

In [43]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,TargetEncoder,StandardScaler

In [39]:
huge_cols = [key for key, val in categorical_data_unique.items() if val[0] > 50]

small_cols = [key for key, val in categorical_data_unique.items() if val[0] <= 50]

In [46]:
preprocessor = ColumnTransformer(
    transformers = [
        ("high_cardinality", TargetEncoder(smooth='auto'),huge_cols),
        ("low_cardinality", OneHotEncoder(sparse_output=False,handle_unknown='ignore'),small_cols),
        ('scaling',StandardScaler(),numerical_feat[1:])
    ],
    remainder='drop'
)

X_train_preprocessed=preprocessor.fit_transform(X_train,y_train)
X_test_preprocessed=preprocessor.transform(X_test)



In [ ]:
names = preprocessor.get_feature_names_out()

In [57]:
X_train_encoded_data = pd.DataFrame(X_train_preprocessed,columns=names)
X_test_encoded_data = pd.DataFrame(X_test_preprocessed,columns=names)



In [73]:
X_test_encoded_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Columns: 178 entries, high_cardinality__categorical_feature_1 to scaling__integer_feature_13
dtypes: float64(178)
memory usage: 10.9 MB


In [66]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import datetime

In [74]:
model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train_encoded_data.shape[1],)),
    Dense(32,activation='relu'),
    Dense(16,activation='relu'),
    Dense(1,activation='sigmoid')
]
)

c:\Users\krish\anaconda3\envs\pytorch_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [75]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │        11,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,081 (55.00 KB)

 Trainable params: 14,081 (55.00 KB)

 Non-trainable params: 0 (0.00 B)

In [76]:
## Compile the model
model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

In [77]:
## Set up Early Stopping

early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [81]:
history = model.fit(X_train_encoded_data,y_train,epochs=100,validation_data=(X_test_encoded_data,y_test),callbacks=[early_stopping_callback])

Epoch 1/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9677 - loss: 0.1338 - val_accuracy: 0.9694 - val_loss: 0.1297
Epoch 2/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9676 - loss: 0.1323 - val_accuracy: 0.9694 - val_loss: 0.1305
Epoch 3/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9677 - loss: 0.1306 - val_accuracy: 0.9694 - val_loss: 0.1321
Epoch 4/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9677 - loss: 0.1280 - val_accuracy: 0.9694 - val_loss: 0.1378
Epoch 5/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9676 - loss: 0.1259 - val_accuracy: 0.9694 - val_loss: 0.1354
Epoch 6/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9677 - loss: 0.1231 - val_accuracy: 0.9694 - val_loss: 0.1390
Epoch 7/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9677 - loss: 0.1201 - val_accuracy: 0.9694 - val_loss: 0.1426
Epoch 8/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9679 - loss: 0

In [79]:
print("X_train:", X_train_encoded_data.shape)
print("X_test:", X_test.shape)
print("Model input:", model.input_shape)

X_train: (32000, 178)
X_test: (8000, 39)
Model input: (None, 178)


In [82]:
data_test = pd.read_csv('test.csv')

In [105]:
X_dtest = data_test.drop(columns=['label'])
y_dtest = data_test['label']

In [106]:
data_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 40 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   label                   10000 non-null  int64  
 1   integer_feature_1       10000 non-null  float64
 2   integer_feature_2       10000 non-null  float64
 3   integer_feature_3       10000 non-null  float64
 4   integer_feature_4       10000 non-null  float64
 5   integer_feature_5       10000 non-null  float64
 6   integer_feature_6       10000 non-null  float64
 7   integer_feature_7       10000 non-null  float64
 8   integer_feature_8       10000 non-null  int64  
 9   integer_feature_9       10000 non-null  int64  
 10  integer_feature_10      10000 non-null  float64
 11  integer_feature_11      10000 non-null  float64
 12  integer_feature_12      10000 non-null  float64
 13  integer_feature_13      10000 non-null  float64
 14  categorical_feature_1   10000 non-null  str    
 1

In [107]:
categorical_testfeat = [i for i in data_test.columns if data_test[i].dtype == 'str']
numerical_testfeat = [i for i in data_test.columns if data_test[i].dtype == float or data_test[i].dtype == int]

In [108]:
## preprocessing and handle the NAN values

## I put median value from NAN in integer_feature and max occurance for categorical_features

for i in numerical_testfeat:
    data_test[i] = data_test[i].fillna(data_test[i].median())


for i in categorical_testfeat:
    data_test[i] = data_test[i].fillna(data_test[i].mode()[0])



In [109]:
X_dtest_encoded = preprocessor.transform(X_dtest)

In [110]:
y_pred = model.predict(X_dtest_encoded)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 831us/step


In [111]:
y_pred

array([[0.02648838],
       [0.04215134],
       [0.02200862],
       ...,
       [0.02156237],
       [0.022054  ],
       [0.04300215]], dtype=float32)

In [115]:
y_dtest = data_test['label']

In [119]:
y_true = data_test['label']

In [121]:
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    average_precision_score,
    log_loss,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay
)
import matplotlib.pyplot as plt



# Default threshold
threshold = 0.5
y_pred_class = (y_pred >= threshold).astype(int)

# Metrics
roc_auc = roc_auc_score(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred_class)
pr_auc = average_precision_score(y_true, y_pred)
logloss = log_loss(y_true, y_pred)
f1 = f1_score(y_true, y_pred_class)
precision = precision_score(y_true, y_pred_class)
recall = recall_score(y_true, y_pred_class)

print("ROC-AUC  :", roc_auc)
print("Accuracy :", accuracy)
print("PR-AUC   :", pr_auc)
print("Log Loss :", logloss)
print("F1 Score :", f1)
print("Precision:", precision)
print("Recall   :", recall)

print("\nClassification Report:")
print(classification_report(y_true, y_pred_class))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred_class))

ROC-AUC  : 0.7129982781385366
Accuracy : 0.9665
PR-AUC   : 0.08367227650050578
Log Loss : 0.1373307704925537
F1 Score : 0.0
Precision: 0.0
Recall   : 0.0

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      9665
           1       0.00      0.00      0.00       335

    accuracy                           0.97     10000
   macro avg       0.48      0.50      0.49     10000
weighted avg       0.93      0.97      0.95     10000


Confusion Matrix:
[[9665    0]
 [ 335    0]]


c:\Users\krish\anaconda3\envs\pytorch_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\krish\anaconda3\envs\pytorch_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\krish\anaconda3\envs\pytorch_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", re

In [122]:
## with weights

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight = dict(zip(classes, weights))

print(class_weight)

{np.int64(0): np.float64(0.516712417245277), np.int64(1): np.float64(15.458937198067632)}


In [123]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [125]:
history = model.fit(
    X_train_encoded_data,
    y_train,
    validation_data=(X_test_encoded_data, y_test),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping_callback],
    class_weight=class_weight
)

Epoch 1/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7324 - loss: 0.6737 - val_accuracy: 0.5786 - val_loss: 0.6471
Epoch 2/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6343 - loss: 0.6175 - val_accuracy: 0.6904 - val_loss: 0.6055
Epoch 3/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6557 - loss: 0.5979 - val_accuracy: 0.7188 - val_loss: 0.5194
Epoch 4/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6672 - loss: 0.5820 - val_accuracy: 0.6210 - val_loss: 0.6357
Epoch 5/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6718 - loss: 0.5603 - val_accuracy: 0.5751 - val_loss: 0.6780
Epoch 6/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6882 - loss: 0.5392 - val_accuracy: 0.6275 - val_loss: 0.6370
Epoch 7/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7038 - loss: 0.5187 - val_accuracy: 0.7418 - val_loss: 0.5068
Epoch 8/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7208 - loss: 0

In [126]:
y_pred = model.predict(X_dtest_encoded)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 816us/step


In [127]:
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    average_precision_score,
    log_loss,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay
)
import matplotlib.pyplot as plt



# Default threshold
threshold = 0.5
y_pred_class = (y_pred >= threshold).astype(int)

# Metrics
roc_auc = roc_auc_score(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred_class)
pr_auc = average_precision_score(y_true, y_pred)
logloss = log_loss(y_true, y_pred)
f1 = f1_score(y_true, y_pred_class)
precision = precision_score(y_true, y_pred_class)
recall = recall_score(y_true, y_pred_class)

print("ROC-AUC  :", roc_auc)
print("Accuracy :", accuracy)
print("PR-AUC   :", pr_auc)
print("Log Loss :", logloss)
print("F1 Score :", f1)
print("Precision:", precision)
print("Recall   :", recall)

print("\nClassification Report:")
print(classification_report(y_true, y_pred_class))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred_class))

ROC-AUC  : 0.6171234875802055
Accuracy : 0.8077
PR-AUC   : 0.055724426052487425
Log Loss : 0.42097339034080505
F1 Score : 0.09845288326300984
Precision: 0.05839822024471635
Recall   : 0.31343283582089554

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.82      0.89      9665
           1       0.06      0.31      0.10       335

    accuracy                           0.81     10000
   macro avg       0.52      0.57      0.50     10000
weighted avg       0.94      0.81      0.87     10000


Confusion Matrix:
[[7972 1693]
 [ 230  105]]
